# 0.0 Setup and Keys

**AIME-Con 2026 · Generative AI for Item Development**

**What you'll do**
1. Get the API keys (Google AI Studio, Groq, Huggingface, Tavily).
2. Store them in Colab Secrets.
3. Install the workshop stack.
4. Run a smoke test that proves both providers answer.

| | |
|---|---|
| **Runtime needed** | CPU (no GPU) |
| **Time** | ~15 minutes |
| **Keys needed** | `GOOGLE_API_KEY`, `GROQ_API_KEY`, `HF_TOKEN`. `TAVILY_API_KEY` optional. |


##  Get your keys

**1. `GOOGLE_API_KEY` — Google AI Studio**
- Go to **https://aistudio.google.com/**, sign in with a Google account.
- Click **Get API key** → **Create API key**.
- Copy the key and paste in a secure notebook or word document. This is the free tier of the Gemini API.

**2. `GROQ_API_KEY` — Groq Console**
- Go to **https://console.groq.com/home**, sign in.
- Open **API Keys** → **Create API Key**, name it something like `aime-workshop`.
- Copy it now and paste in a secure notebook or word document. Groq shows the key once and never again.

**3. `TAVILY_API_KEY` — optional, agent notebook only**
- **https://www.tavily.com/** → sign up → copy the key from the dashboard and paste in a secure notebook or word document.

**4. `HF_TOKEN` - Huggingface**
- Visit **https://huggingface.co/** and sign up
- Generate Token
    - Go to Settings > Access Tokens
- Click “Create new token”, choose “read” access, give it a token name, and click “Create token”.
- Copy the provided key to a secure notebook or word document.

A note on institutional accounts: if your work Google account is locked down by an
administrator, AI Studio may refuse to issue a key. Use a personal Google account for the workshop.

##  Store the keys in Colab Secrets

Do **not** paste a key into a code cell. Notebooks get shared, screenshotted, and committed to
repos; a key pasted into a cell travels with the file. Colab Secrets keeps the value outside the
notebook.

1. Click the **key icon** (🔑) in the left sidebar of Colab.
2. Click **+ Add new secret**.
3. Name it exactly `GOOGLE_API_KEY`, paste the value that you copied in your secure note into **Value**.
4. Toggle **Notebook access** on for this notebook.
5. Repeat for `GROQ_API_KEY`, `TAVILY_API_KEY`, and `HF_TOKEN`.

Names are case-sensitive and must match exactly. The secrets are attached to
your Google account, so every notebook in this workshop will find them.

In [1]:
# Install the workshop stack
%pip install -q -U langchain langchain-google-genai langchain-groq

print("Install finished.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 kB 10.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.60.0 which is incompatible.
Install finished.


In [2]:
# Load the keys from Colab Secrets into environment variables
import os
from google.colab import userdata


def load_key(name, required=True):
    """Copy one secret from Colab Secrets into os.environ. Returns True if found."""
    try:
        value = userdata.get(name)
    except Exception:
        value = None

    if value:
        os.environ[name] = value
        print(f"{name}: OK")
        return True

    if required:
        print(
            f"{name}: NOT FOUND\n"
            f"  Fix: click the key icon in the left sidebar, add a secret named exactly\n"
            f"  '{name}', paste your key as the value, and turn ON 'Notebook access'.\n"
            f"  Then re-run this cell."
        )
    else:
        print(f"{name}: not set (optional - only needed for the agent notebook)")
    return False


have_google = load_key("GOOGLE_API_KEY")
have_groq = load_key("GROQ_API_KEY")
have_tavily = load_key("TAVILY_API_KEY", required=False)
have_hf = load_key("HF_TOKEN")

GOOGLE_API_KEY: OK
GROQ_API_KEY: OK
TAVILY_API_KEY: not set (optional - only needed for the agent notebook)
HF_TOKEN: OK


## Which models can my key see?

Model IDs change. Providers retire them, rename them, and move them in and out of the free tier
several of the IDs used in the 2025 edition of this workshop no longer resolve. Rather than trust a
constant written months ago, ask your key what it can reach at the moment.

The cell below lists every model your `GOOGLE_API_KEY` can call for ordinary text generation
(`generateContent`). If a model ID in a later notebook throws a 404, come back here, find a
current ID, and change the one constant at the top of that notebook.

In [3]:
# Ask the API which models this key supports for generateContent.
try:
    from google import genai  # installed as a dependency of langchain-google-genai

    client = genai.Client()  # reads GOOGLE_API_KEY from the environment
    names = [
        m.name
        for m in client.models.list()
        if m.supported_actions and "generateContent" in m.supported_actions
    ]
    # Keep the list readable: current Gemini text models only.
    shortlist = sorted(n.replace("models/", "") for n in names if "gemini" in n)

    print(f"{len(names)} models support generateContent. Gemini text models:\n")
    for n in shortlist:
        print(" ", n)

except Exception as e:
    print("Could not list models.")
    print("Likely cause: GOOGLE_API_KEY is missing, mistyped, or Notebook access is off.")
    print(f"Details: {type(e).__name__}: {e}")

44 models support generateContent. Gemini text models:

  gemini-2.5-computer-use-preview-10-2025
  gemini-2.5-flash
  gemini-2.5-flash-image
  gemini-2.5-flash-lite
  gemini-2.5-flash-preview-tts
  gemini-2.5-pro
  gemini-2.5-pro-preview-tts
  gemini-3-flash-preview
  gemini-3-pro-image
  gemini-3-pro-image-preview
  gemini-3.1-flash-image
  gemini-3.1-flash-image-preview
  gemini-3.1-flash-lite
  gemini-3.1-flash-lite-image
  gemini-3.1-flash-lite-preview
  gemini-3.1-flash-tts-preview
  gemini-3.1-pro-preview
  gemini-3.1-pro-preview-customtools
  gemini-3.5-flash
  gemini-3.5-flash-lite
  gemini-3.5-transcribe
  gemini-3.6-flash
  gemini-3.7-flash
  gemini-3.8-flash
  gemini-3.8-flash-lite-tts
  gemini-3.8-flash-tts
  gemini-flash-latest
  gemini-flash-lite-latest
  gemini-omni-1.1-flash
  gemini-omni-flash-preview
  gemini-pro-latest
  gemini-robotics-er-2-preview


In [6]:
# Smoke test: one call to each provider. Each should print a single line.
GEMINI_MODEL = "gemini-3.8-flash"
GROQ_MODEL = "openai/gpt-oss-120b"

PROMPT = "In one sentence, what makes a multiple-choice distractor plausible?"


def diagnose(e):
    """Turn an exception into the most likely cause, in plain language."""
    msg = str(e).lower()
    if "429" in msg or "quota" in msg or "rate limit" in msg:
        return "Free-tier quota or rate limit hit. Wait 60 seconds and re-run."
    if "404" in msg or "not found" in msg:
        return "Model ID not found - it was probably retired. Use the model list cell above."
    if "401" in msg or "403" in msg or "api key" in msg or "permission" in msg:
        return "Key problem: missing, mistyped, or Notebook access not toggled on."
    return "Unexpected error - see the troubleshooting table below."


# --- Gemini ---
try:
    from langchain_google_genai import ChatGoogleGenerativeAI

    gemini = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0)
    print("Gemini:", gemini.invoke(PROMPT).text.strip()[:300])
except Exception as e:
    print(f"Gemini FAILED. {diagnose(e)}")
    print(f"  Details: {type(e).__name__}: {e}")

print()

# --- Groq ---
try:
    from langchain_groq import ChatGroq

    groq = ChatGroq(model=GROQ_MODEL, temperature=0)
    print("Groq:", groq.invoke(PROMPT).content.strip()[:300])
except Exception as e:
    print(f"Groq FAILED. {diagnose(e)}")
    print(f"  Details: {type(e).__name__}: {e}")

Gemini: A multiple-choice distractor is plausible when it targets common misconceptions, flawed reasoning, or partial knowledge while closely mirroring the correct answer in tone, complexity, and grammatical structure.

Groq: A plausible multiple‑choice distractor is an answer choice that reflects a common misconception, partial truth, or superficially correct detail that appears reasonable to someone who hasn’t mastered the material.
